In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Hyperparameters

The current set of hyperparameters for our model implementation is listed below:

In [2]:
# new hyperparameters
n_layer = 6                  # number of Block layers
dropout = 0.2                # residual dropout          

# hyperparameters
#batch_size = 64             # how many independent sequences to parallel-process?
batch_size = 16              # how many independent sequences to parallel-process?
max_context_size = 256       # "block_size" ... maximum 256 chars context length for predictions
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
loss_estimation_iters = 200  # "eval_iters" ... num. iters for estimate_loss
embedding_size = 384         # "n_embd" ... size of the embedding tensors
num_heads = 6
head_size = embedding_size // num_heads           # num. channels per head of attention

<span style="background-color:#27F5EB49">Don't use `batch_size = 64` unless you have access to an enviroment with at least a proper GPU!</span>

----

## "Magical" Helper

In [3]:
from helper import *

def get_batch(split):
    data = training_data if split == 'train' else validation_data
    ix = torch.randint(len(data) - max_context_size, (batch_size,))
    x = torch.stack([data[i:i+max_context_size] for i in ix]).to(device)
    y = torch.stack([data[i+1:i+max_context_size+1] for i in ix]).to(device)
    return x,y

----

# Final Implementation

## Key Points

* Add allowances to compute per-node learning after the node-to-node communication (see the project-upward $\rightarrow$ non-linearity $\rightarrow$ project-downward in `FeedForward`)
* Add allowances for optimization for scaling up the model training (residual connections, layer normalization)
* Add allowances for regularization for improving model's ability to generalize when presented with unseen data


### `Head`

#### Regularization

In `Head`, regularization via dropout is done in the following line:

    wei = self.dropout(wei) 

This is regularization of the token-to-token communication, namely on the _attention weights_. During training, this randomly suppresses some of the communication links between tokens.

In [4]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(embedding_size, head_size, bias=False)
        self.query = nn.Linear(embedding_size, head_size, bias=False)
        self.value = nn.Linear(embedding_size, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(max_context_size, max_context_size))
        )

        # dropout for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape

        k = self.key(x)            # (B,T,C)
        q = self.query(x)          # (B,T,C)

        wei = q @ k.transpose(-2,-1) * C**-0.5   # (B,T,C) @ (B,C,T) ----> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))     # (B,T,T)
        wei = F.softmax(wei, dim=-1)                                     # (B,T,T)
        wei = self.dropout(wei)    # (B,T,C)
        v = self.value(x)          # (B,T,C)
        out = wei @ v              # (B,T,T) @ (B,T,C) ----> (B,T,C)

        return out

### `MultiHeadAttention`

Encapsulate the logic for multi-head self-attention, abstracting it away from `Block`.

#### Regularization

In `MultiHeadAttention`,  regularization is done in the following line:

    out = self.dropout(self.proj(out))

This is regularization of the _combined multi-head attention sublayer output_. Given the following line in `Block`:

    x = x + self.sa(self.ln1(x))

... what this means is that at the very end of every `MultiHeadAttention` sublayer in a Transformer block, _residual-path dropout_ regularizes what attention contributes to the residual stream.

Please refer to the Residual Dropout section in 5.4. Regularization, page 8 in [Attention Is All You Need](https://arxiv.org/pdf/1706.03762).

In [5]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(embedding_size, embedding_size)

        # dropout for regularization
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

### `FeedForward`

Encapsulates the logic for the Multi-Layer Perceptron. The MLP follows the node-to-node communication of self-attention with per-node computation that provides computational capacity to learn transformations of each token's representation

Please see 3.3 Position-wise Feed-Forward Networks, page 5 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762)

> While the linear transformations are the same across different positions, they use different parameters
from layer to layer. Another way of describing this is as two convolutions with kernel size 1.
<span style="background-color:#27F5EB49">The dimensionality of input and output is $d_{\mathrm{model}} = 512$, and the inner-layer has dimensionality
$d_{\mathrm{ff}} = 2048$.</span>

#### Regularization

After the MLP processing, , _residual-path dropout_ regularizes what the MLP contributes to the residual stream.

In [6]:
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, embedding_size):
        super().__init__()

        self.net = nn.Sequential(

            # this is the MLP here!
            nn.Linear(embedding_size, 4 * embedding_size),
            nn.ReLU(),
            nn.Linear(4 * embedding_size, embedding_size),

            # dropout for regularization, after the MLP
            nn.Dropout(dropout),

        )

    def forward(self, x):
        return self.net(x)

### `Block`

Implements a _pre-norm arranged block_ in the Transformer architecture.

A `Block` encapsulates the multi-headed attention; a feed-forward sublayer; and the Add & Norm parts of a Transformer. 

N.B., the Add & Norm logic bits in Karpathy's nanoGPT appear _before_ both the multi-head self-attention sublayer and again _before_ the feed-forward sublayer.

#### Optimization

Please see the Decoder section in 3.1 Encoder and Decoder Stacks, page 3 of [Attention Is All You Need](https://arxiv.org/pdf/1706.03762)

> <strong>Decoder</strong>: The decoder is also composed of a stack of $N = 6$ identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack. Similar to the encoder, <span style="background-color:#27F5EB49">we employ residual connections
around each of the sub-layers, followed by layer normalization</span>. We also modify the self-attention
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position $i$ can depend only on the known outputs at positions less than $i$.


The following line implements _pre-norm arranged `LayerNorm`_ on the self-attention input in the residual pathway _before_ the addition in the residual connection.

    x = x + self.sa(self.ln1(x)) 

Likewise, the subsequent line implements _pre-norm arranged `LayerNorm`_ on the feed-forward input in the residual pathway _before_ the addition in the residual connection.

    x = x + self.ffwd(self.ln2(x))

In [7]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, embedding_size, num_heads):
        # embedding_size (aka n_embd): size of token embeddings; num_heads: number of attention heads
        super().__init__()

        head_size =  embedding_size // num_heads
        self.sa = MultiHeadAttention(num_heads, head_size)
        self.ffwd = FeedForward(embedding_size)

        # regularization via LayerNorm
        self.ln1 = nn.LayerNorm(embedding_size)
        self.ln2 = nn.LayerNorm(embedding_size)

    def forward(self, x):
        # optimization via residual connection
        x = x + self.sa(self.ln1(x))    # pre-norm arrangement normalizes input to self-attention sublayer
        x = x + self.ffwd(self.ln2(x))  # pre-norm arrangement normalizes input to feed-forward sublayer
        return x

### `BigramLanguageModel`

This encapsulate multiple `Block`s of multi-head self-attention; a final layer normalization for optimization after the last `Block` before hitting the output head; and the final linear layer for the output head.

In [8]:
# final implementation of bigram language model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, embedding_size)
        self.position_embedding_table = nn.Embedding(max_context_size, embedding_size)

        # now using multiple Blocks for multiple communication & computation 
        self.blocks = nn.Sequential(*[
            Block(embedding_size, num_heads=num_heads)
            for _ in range(n_layer)
        ])

        # final layer normalization for optimization
        self.ln_f = nn.LayerNorm(embedding_size)

        self.lm_head = nn.Linear(embedding_size, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape

        # idx and targets are both (B,T) tensor of int
        tok_embeddings = self.token_embedding_table(idx)  # (B,T,C)
        pos_embeddings = self.position_embedding_table(torch.arange(T, device=device))  # (T,C)

        x = tok_embeddings + pos_embeddings  # (B,T,C)
        x = self.blocks(x)                   # (B,T,C)
        x = self.ln_f(x)                     # (B,T,C) ... 
        logits = self.lm_head(x)             # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) tensor of indices in the current context
        for _ in range(max_new_tokens):

            # crop idx to the last max_context_size tokens
            idx_cond = idx[:, -max_context_size:]

            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]           # becomes (B,C)
            probs = F.softmax(logits, dim=-1)   # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            idx = torch.cat((idx, idx_next), dim=1)  # (B,T+1)
        return idx

## The Training Loop

In [9]:
%%time

torch.manual_seed(1337)

model = BigramLanguageModel().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(loss_estimation_iters)
        for k in range(loss_estimation_iters):
            X,Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss is {losses['train']:.4f}, val loss is {losses['val']:.4f}")

    xb,yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"final loss: {loss.item():.4f}\n")

step 0: train loss is 4.2840, val loss is 4.2821
step 500: train loss is 2.2635, val loss is 2.2986
step 1000: train loss is 1.8469, val loss is 1.9680
step 1500: train loss is 1.6571, val loss is 1.8219
step 2000: train loss is 1.5517, val loss is 1.7346
step 2500: train loss is 1.4750, val loss is 1.6744
step 3000: train loss is 1.4177, val loss is 1.6299
step 3500: train loss is 1.3697, val loss is 1.5902
step 4000: train loss is 1.3419, val loss is 1.5748
step 4500: train loss is 1.3100, val loss is 1.5494
final loss: 1.3349

CPU times: user 39min 44s, sys: 5min 21s, total: 45min 5s
Wall time: 45min 13s


In [10]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.int, device=device)
print(decode(
    model.generate(
        context,
        max_new_tokens=2000
    )[0].tolist()
))


That what ever 'tis press'd time.' Or hat?
Yen my calms sbut my brother, which therein spent,
To she may wail woe swarce me honour,
And how I bride unctogething it and lutches?

DUKE VINCENTIO:
If the present hours officerable next from,
And the predecitious things unto by my son,
And blitter it see and best no take thine. The Englis
Torme, but take that crep oF York, graci?

DUKE VINCENTIO:
O queen witness
'Tis bound filly hat, though thought in their troyalts
And I the head.
Had you this pantrat trawning reveng in blood
Throught of all sensest procion: she is before
Shalt ince our sworments ne'er son will year teath,
Which one in a bid infall woard th do break;
'by was he daughter will signor, and what it again,
Now my plain, that know's tears, from this bubsides.
As a sonver's never children; we then have beard
Show's harm sister'd his friery face,
But the land in his son hearest sides:
Made desire mi word, dread
On move sister somes,
And my lord; or quices to his ague,
In every th

----

## Count the Model Weights

How many parameters does this model implementation have?

In [11]:
c = 0
for name, p in model.named_parameters():
    print(f"{name:60s} {p.numel():>10,}")
    c += p.numel()

print(f"{''.join(['-']*71)}")
print(f"{'total parameters':60s} {c:>10,}")

token_embedding_table.weight                                     24,960
position_embedding_table.weight                                  98,304
blocks.0.sa.heads.0.key.weight                                   24,576
blocks.0.sa.heads.0.query.weight                                 24,576
blocks.0.sa.heads.0.value.weight                                 24,576
blocks.0.sa.heads.1.key.weight                                   24,576
blocks.0.sa.heads.1.query.weight                                 24,576
blocks.0.sa.heads.1.value.weight                                 24,576
blocks.0.sa.heads.2.key.weight                                   24,576
blocks.0.sa.heads.2.query.weight                                 24,576
blocks.0.sa.heads.2.value.weight                                 24,576
blocks.0.sa.heads.3.key.weight                                   24,576
blocks.0.sa.heads.3.query.weight                                 24,576
blocks.0.sa.heads.3.value.weight                                